# Experiment Q3 — Audio Filtering Using Low-Pass and Band-Pass Filters

### Q3(a)
Select an MP3 song and process it using a low-pass filter with at least three different cutoff frequencies. Observe the effect on clarity, tonal balance, and overall character.

### Q3(b)
Use an appropriate band-pass filter to isolate the main vocal-frequency region and estimate the remaining/background component. Save each result as an MP3 file and analyze the effect of filtering.

**Input file:** `Original.mp3` (the notebook also provides an upload option if the file is not already present.)

In [ ]:
# ============================================================
# 1. INSTALL / VERIFY REQUIRED PACKAGES
# ============================================================

!pip -q install librosa soundfile scipy matplotlib pydub
!apt-get -qq update
!apt-get -qq install ffmpeg

print('Required packages are ready.')

In [ ]:
# ============================================================
# 2. IMPORT LIBRARIES AND FIND / UPLOAD THE MP3 FILE
# ============================================================

import os
import subprocess
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import soundfile as sf
from scipy.signal import butter, sosfiltfilt
from IPython.display import Audio, display

INPUT_FILE = '/content/Original.mp3'

if not os.path.exists(INPUT_FILE):
    from google.colab import files
    print('Original.mp3 was not found. Please select your MP3 file.')
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError('No audio file was uploaded.')
    uploaded_name = next(iter(uploaded))
    uploaded_path = os.path.join('/content', uploaded_name)
    if uploaded_name != 'Original.mp3':
        os.replace(uploaded_path, INPUT_FILE)

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError('Could not find /content/Original.mp3.')

print('Input file:', INPUT_FILE)
print('File size:', round(os.path.getsize(INPUT_FILE) / (1024 * 1024), 2), 'MB')

In [ ]:
# ============================================================
# 3. LOAD AUDIO
# ============================================================

y, sr = librosa.load(INPUT_FILE, sr=None, mono=True)

duration = len(y) / sr
nyquist = sr / 2

print('Audio loaded successfully!')
print(f'Sample Rate : {sr} Hz')
print(f'Duration    : {duration:.2f} seconds')
print(f'Nyquist Freq: {nyquist:.0f} Hz')

In [ ]:
# ============================================================
# 4. FILTER FUNCTIONS
# ============================================================

def lowpass_filter(signal, cutoff, sample_rate, order=6):
    if not 0 < cutoff < sample_rate / 2:
        raise ValueError(f'Cutoff must be between 0 and {sample_rate/2:.0f} Hz.')
    sos = butter(
        order,
        cutoff,
        btype='lowpass',
        fs=sample_rate,
        output='sos'
    )
    return sosfiltfilt(sos, signal)

def bandpass_filter(signal, lowcut, highcut, sample_rate, order=6):
    if not 0 < lowcut < highcut < sample_rate / 2:
        raise ValueError('Invalid band-pass frequency range.')
    sos = butter(
        order,
        [lowcut, highcut],
        btype='bandpass',
        fs=sample_rate,
        output='sos'
    )
    return sosfiltfilt(sos, signal)

def plot_spectrogram(signal, sample_rate, title, max_freq=None):
    D = librosa.stft(signal, n_fft=2048, hop_length=512)
    D_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
    plt.figure(figsize=(12, 4))
    librosa.display.specshow(
        D_db,
        sr=sample_rate,
        hop_length=512,
        x_axis='time',
        y_axis='hz'
    )
    if max_freq is not None:
        plt.ylim(0, min(max_freq, sample_rate / 2))
    plt.colorbar(format='%+2.0f dB')
    plt.title(title)
    plt.xlabel('Time (seconds)')
    plt.ylabel('Frequency (Hz)')
    plt.tight_layout()
    plt.show()

print('Filter functions are ready.')

## Q3(a) — Low-Pass Filter

Three cutoff frequencies are used: **1000 Hz, 3000 Hz, and 6000 Hz**. Lower cutoff frequencies remove more high-frequency content, while higher cutoff frequencies preserve more of the original tonal detail.

In [ ]:
# ============================================================
# 5. Q3(a) — APPLY THREE LOW-PASS FILTERS
# ============================================================

cutoffs = [1000, 3000, 6000]
lowpass_results = {}

for cutoff in cutoffs:
    lowpass_results[cutoff] = lowpass_filter(y, cutoff, sr)
    print(f'✓ Low-pass filter applied: {cutoff} Hz')

print('\nQ3(a) filtering completed.')

In [ ]:
# ============================================================
# 6. Q3(a) — SPECTROGRAMS
# ============================================================

plot_spectrogram(y, sr, 'Original Audio - Full Spectrum')

for cutoff in cutoffs:
    plot_spectrogram(
        lowpass_results[cutoff],
        sr,
        f'Low-Pass Filter - Cutoff Frequency: {cutoff} Hz'
    )

In [ ]:
# ============================================================
# 7. Q3(a) — SAVE LOW-PASS RESULTS AS MP3
# ============================================================

def save_as_mp3(signal, sample_rate, output_mp3, bitrate='192k'):
    temp_wav = output_mp3.replace('.mp3', '_temp.wav')
    sf.write(temp_wav, signal, sample_rate, subtype='PCM_16')
    subprocess.run(
        ['ffmpeg', '-y', '-loglevel', 'error', '-i', temp_wav,
         '-codec:a', 'libmp3lame', '-b:a', bitrate, output_mp3],
        check=True
    )
    os.remove(temp_wav)

for cutoff in cutoffs:
    output = f'/content/lowpass_{cutoff}Hz.mp3'
    save_as_mp3(lowpass_results[cutoff], sr, output)
    print('✓ Saved:', output)

In [ ]:
# ============================================================
# 8. OPTIONAL LISTENING — Q3(a)
# ============================================================

for cutoff in cutoffs:
    print(f'Low-pass {cutoff} Hz:')
    display(Audio(f'/content/lowpass_{cutoff}Hz.mp3'))

## Q3(a) Observations

- **1000 Hz:** Strong attenuation of high frequencies. The song sounds muffled and bass/midrange dominated; fine vocal and instrument details are reduced.
- **3000 Hz:** More vocal intelligibility and midrange detail are retained, but cymbals and other high-frequency components are noticeably reduced.
- **6000 Hz:** Most of the important musical and vocal detail is retained while the highest-frequency content is attenuated. The result is closer to the original song.

**Conclusion for (a):** Increasing the low-pass cutoff allows a wider frequency range to pass, improving clarity and preserving more of the song's original tonal balance.

## Q3(b) — Band-Pass Vocal-Region Filtering and Background Estimation

A **300–3400 Hz Butterworth band-pass filter** is used as an approximate vocal/speech-frequency band. The band-pass output emphasizes frequencies in this region. The estimated background is obtained as the residual `original − vocal-band`, so it should be described as an **estimated background component**, not perfect source separation.

In [ ]:
# ============================================================
# 9. Q3(b) — BAND-PASS VOCAL REGION
# ============================================================

lowcut = 300
highcut = 3400

vocal_audio = bandpass_filter(
    y,
    lowcut,
    highcut,
    sr
)

# Residual signal: an estimate of the remaining/background component
background_audio = y - vocal_audio

print(f'Band-pass range: {lowcut}–{highcut} Hz')
print('✓ Estimated vocal-frequency signal created.')
print('✓ Estimated background/residual signal created.')

In [ ]:
# ============================================================
# 10. Q3(b) — SPECTROGRAMS
# ============================================================

plot_spectrogram(
    vocal_audio,
    sr,
    'Estimated Vocal Frequencies - Band-Pass 300–3400 Hz',
    max_freq=8000
)

plot_spectrogram(
    background_audio,
    sr,
    'Estimated Background / Residual Component',
    max_freq=8000
)

In [ ]:
# ============================================================
# 11. Q3(b) — SAVE VOCAL AND BACKGROUND RESULTS AS MP3
# ============================================================

vocal_output = '/content/filtered_vocals.mp3'
background_output = '/content/estimated_background.mp3'

save_as_mp3(vocal_audio, sr, vocal_output)
save_as_mp3(background_audio, sr, background_output)

print('✓ Saved:', vocal_output)
print('✓ Saved:', background_output)

In [ ]:
# ============================================================
# 12. OPTIONAL LISTENING — Q3(b)
# ============================================================

print('Estimated vocal-frequency output:')
display(Audio(vocal_output))

print('Estimated background/residual output:')
display(Audio(background_output))

In [ ]:
# ============================================================
# 13. WAVEFORM COMPARISON
# ============================================================

plt.figure(figsize=(12, 8))

plt.subplot(3, 1, 1)
librosa.display.waveshow(y, sr=sr)
plt.title('Original Audio')
plt.xlabel('Time (seconds)')

plt.subplot(3, 1, 2)
librosa.display.waveshow(vocal_audio, sr=sr)
plt.title('Estimated Vocal-Frequency Signal')
plt.xlabel('Time (seconds)')

plt.subplot(3, 1, 3)
librosa.display.waveshow(background_audio, sr=sr)
plt.title('Estimated Background / Residual Signal')
plt.xlabel('Time (seconds)')

plt.tight_layout()
plt.show()

## Q3(b) Observations

- The **300–3400 Hz band-pass output** emphasizes the frequency region in which speech and many vocal components are concentrated. Vocals become more prominent, while bass and very high-frequency components are attenuated.
- The **estimated background/residual** contains the frequency content outside the emphasized vocal band and may still contain some vocal energy because real vocals and instruments overlap in frequency.
- Therefore, the band-pass method is useful for demonstrating frequency-selective filtering, but it is **not equivalent to professional vocal/source separation**.

**Conclusion for (b):** Band-pass filtering can emphasize a selected vocal-frequency region and the residual can be used as an approximate background component. The result depends on the overlap between vocal and instrumental frequencies.

In [ ]:
# ============================================================
# 14. FINAL OUTPUT CHECK
# ============================================================

output_files = [
    '/content/lowpass_1000Hz.mp3',
    '/content/lowpass_3000Hz.mp3',
    '/content/lowpass_6000Hz.mp3',
    '/content/filtered_vocals.mp3',
    '/content/estimated_background.mp3'
]

print('=' * 60)
print('EXPERIMENT COMPLETED SUCCESSFULLY')
print('=' * 60)

for file in output_files:
    if os.path.exists(file):
        size_mb = os.path.getsize(file) / (1024 * 1024)
        print(f'✓ {os.path.basename(file):35s} {size_mb:.2f} MB')
    else:
        print(f'✗ Missing: {file}')

In [ ]:
# ============================================================
# 15. OPTIONAL — ZIP ALL MP3 OUTPUTS FOR DOWNLOAD
# ============================================================

zip_path = '/content/Q3_Audio_Filtering_Outputs.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for file in output_files:
        if os.path.exists(file):
            zf.write(file, arcname=os.path.basename(file))

print('ZIP created:', zip_path)

from google.colab import files

# Uncomment the next line if you want Colab to download the ZIP automatically.
# files.download(zip_path)